In [ ]:
SEED = 42
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(SEED)


# Task 02: EDA and Insight Generation
This notebook uses only `../task_01/outputs/ingested.csv` and saves all outputs to `outputs/`.

In [ ]:
INPUT_PATH = Path('../task_01/outputs/ingested.csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
df.shape

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['price'], bins=80, color='#2a9d8f', edgecolor='white')
ax.set_title('Price Distribution (Original Scale)')
ax.set_xlabel('Price (USD/night)')
ax.set_ylabel('Listing Count')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'price_distribution_hist.png', dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(np.log1p(df['price']), bins=80, color='#264653', edgecolor='white')
ax.set_title('Price Distribution (log1p for inspection only)')
ax.set_xlabel('log(1 + price)')
ax.set_ylabel('Listing Count')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'price_distribution_log_hist.png', dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
ax.boxplot(df['price'], vert=True, patch_artist=True, boxprops=dict(facecolor='#e9c46a'))
ax.set_title('Price Boxplot (Original Scale)')
ax.set_ylabel('Price (USD/night)')
ax.set_xticks([1])
ax.set_xticklabels(['price'])
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'price_boxplot.png', dpi=160)
plt.close(fig)


## Interpretation: Target Variable (`price`)
The target is heavily right-skewed with a long tail of expensive listings, so raw-scale models can be influenced by extreme values. The log-view is much more compact and suggests that target transformation should be considered in Task 03, but not applied here. Outliers appear genuine market variation rather than simple missing-data artifacts.

In [ ]:
price_cap = df['price'].quantile(0.995)
eda_df = df[df['price'] <= price_cap].copy()

num_cols = [
    'price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month',
    'calculated_host_listings_count', 'availability_365', 'latitude', 'longitude'
]
corr = eda_df[num_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=45, ha='right')
ax.set_yticks(range(len(num_cols)))
ax.set_yticklabels(num_cols)
ax.set_title('Numeric Feature Correlation Heatmap')
fig.colorbar(im, ax=ax, label='Correlation')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'feature_correlation_heatmap.png', dpi=160)
plt.close(fig)


## Interpretation: Feature Correlation
Linear correlations with price are modest, which suggests relationships are likely non-linear and interaction-driven. Coordinates and room/location variables likely carry stronger signal than simple review-volume metrics. This supports trying interaction and non-linear models in later tasks.

In [ ]:
room_types = list(eda_df['room_type'].dropna().unique())
room_data = [eda_df.loc[eda_df['room_type'] == r, 'price'].values for r in room_types]
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(room_data, patch_artist=True)
ax.set_xticks(range(1, len(room_types) + 1))
ax.set_xticklabels(room_types, rotation=20)
ax.set_title('Price by Room Type')
ax.set_xlabel('Room Type')
ax.set_ylabel('Price (USD/night)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'price_by_room_type.png', dpi=160)
plt.close(fig)

boroughs = list(eda_df['neighbourhood_group'].dropna().unique())
borough_data = [eda_df.loc[eda_df['neighbourhood_group'] == b, 'price'].values for b in boroughs]
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(borough_data, patch_artist=True)
ax.set_xticks(range(1, len(boroughs) + 1))
ax.set_xticklabels(boroughs, rotation=20)
ax.set_title('Price by Borough')
ax.set_xlabel('Neighbourhood Group')
ax.set_ylabel('Price (USD/night)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'price_by_borough.png', dpi=160)
plt.close(fig)


## Interpretation: Room Type and Borough Effects
Room type is one of the clearest separators of price, with entire-home listings generally much higher. Borough-level distributions also differ substantially, indicating strong location effects. These two variables should be treated as high-priority predictors and likely interaction terms.

In [ ]:
sample = eda_df.sample(n=min(15000, len(eda_df)), random_state=SEED)
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(sample['longitude'], sample['latitude'], c=sample['price'], cmap='viridis', s=7, alpha=0.5)
ax.set_title('Geographic Pattern of Price')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
fig.colorbar(sc, ax=ax, label='Price (USD/night)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geo_price_scatter.png', dpi=160)
plt.close(fig)

pivot = eda_df.pivot_table(index='neighbourhood_group', columns='room_type', values='price', aggfunc='median')
fig, ax = plt.subplots(figsize=(9, 5))
im2 = ax.imshow(pivot.values, cmap='YlGnBu')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(list(pivot.columns), rotation=20)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(list(pivot.index))
ax.set_title('Median Price by Borough and Room Type')
ax.set_xlabel('Room Type')
ax.set_ylabel('Neighbourhood Group')
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if pd.notna(val):
            ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9)
fig.colorbar(im2, ax=ax, label='Median Price (USD/night)')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'borough_roomtype_median_price_heatmap.png', dpi=160)
plt.close(fig)


## Interpretation: Location-Price Relationships and Subgroups
The geographic scatter reveals clear spatial clustering of expensive listings, consistent with high-demand central areas. The borough x room-type heatmap shows subgroup behavior: combinations such as Manhattan + Entire home/apt are materially different from lower-price segments. This supports subgroup-aware modeling strategies.

## Data Quality Decisions from EDA
Rows above the 99.5th percentile of `price` were removed in `eda_cleaned.csv` to stabilize modeling and reduce leverage from extreme tail observations. This was done without using any train/test split or held-out data and is documented for downstream reproducibility.

In [ ]:
eda_df.to_csv(OUTPUT_DIR / 'eda_cleaned.csv', index=False)

q95 = float(df['price'].quantile(0.95))
q99 = float(df['price'].quantile(0.99))
q995 = float(df['price'].quantile(0.995))
skew = float(df['price'].skew())
corr_price = corr['price'].drop('price').sort_values(key=lambda s: s.abs(), ascending=False)

summary = f'''# EDA Summary (Task 02)

## Target Variable: `price`
- `price` is strongly right-skewed (skewness = {skew:.2f}) with a long tail.
- Key quantiles: 95th={q95:.1f}, 99th={q99:.1f}, 99.5th={q995:.1f}.
- For Task 03, consider robust handling of the target (for example log-transform during training pipeline) and robust metrics.

## Most Important Feature Signals
- `room_type` and `neighbourhood_group` are strong price separators.
- Coordinates (`latitude`, `longitude`) show clear spatial structure.
- Interaction effects (borough x room type) appear important.

## Feature Engineering Ideas for Task 04
- Borough x room_type interactions
- Spatial features from latitude/longitude
- Robust treatment of heavy-tailed numeric variables

## Top Correlations With Price (absolute)
{corr_price.head(6).to_string()}
'''

(OUTPUT_DIR / 'eda_summary.md').write_text(summary, encoding='utf-8')
print('Saved eda_cleaned.csv and eda_summary.md')
